In [1]:
import ROOT
from fitHelper import fit, build_sim_ws, Plotter

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8832950
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x889f6d0


In [2]:
ROOT.EnableImplicitMT(6)

In [3]:
# run_ab = 2
run_ab = 3
# run_ab = 11.2
# split_hel = False
split_hel = True

plot_dir = f"plots/fit_results_{run_ab}ab{'_split_hel' if split_hel else ''}"

In [4]:
# oo_names = [
#     "mlvec_reco",
#     "reco",
#     "mlvec_reco_jm",
#     "reco_jm",
#     "mlvec_clean_reco",
#     "clean_reco",
#     "kinfit_clean_reco",
#     "mlvec_clean_reco_jm",
#     "clean_reco_jm",
#     "kinfit_clean_reco_jm",
#     "mlvec_cheat_clean_reco",
#     "cheat_clean_reco",
#     "mlvec_cheat_clean_reco_jm",
#     "cheat_clean_reco_jm",
#     "mc",
#     "nomb_mc",
#     "nomb_mc_rlep",
#     "nomb_mc_rlep_gamma",
#     "nomb_mc_rlep_brems",
#     "nomb_mc_rlep_cheated_brems",
#     "av_mc",
#     "av_nomb_mc",
#     "av_nomb_mc_rlep",
#     "av_nomb_mc_rlep_gamma",
#     "av_nomb_mc_rlep_cheated_brems",
# ]
oo_names = [
    "mlvec_reco",
    "reco",
    "mlvec_reco_jm",
    "reco_jm",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
    # "kinfit_clean_reco",
    "mlvec_clean_reco_jm",
    "clean_reco_jm",
    "clean_brems_reco_jm",
    "mlvec_clean_brems_reco_jm",
    # "kinfit_clean_reco_jm",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    "mlvec_cheat_clean_reco_jm",
    "cheat_clean_reco_jm",
    "mc",
    "nomb_mc",
    "nomb_mc_rlep",
    "nomb_mc_rlep_gamma",
    "nomb_mc_rlep_brems",
    "nomb_mc_rlep_cheated_brems",
    "nurec_nomb_mc",
    "nurec_nomb_mc_rlep",
    "nurec_nomb_mc_rlep_gamma",
    "nurec_nomb_mc_rlep_brems",
    "nurec_nomb_mc_rlep_cheated_brems",
    "nurec_post94_mc",
    "av_mc",
    "av_nomb_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc_rlep_gamma",
    "av_nomb_mc_rlep_brems",
    "av_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_post94_mc",
]
# input_path = "fit-configs/signal-only-mW-pol"
# input_path = "fit-configs/signal-only-mW-pol-new"
input_path = "fit-configs/signal-only-mW-pol-new2"
oo_name = "mlvec_clean_reco"
run_configurations = {
    "new_share_high_ppol" : [
    # (lumi_share, e_pol, p_pol)
        (0.05, -0.8, -0.6),
        (0.45, -0.8, +0.6),
        (0.45, +0.8, -0.6),
        (0.05, +0.8, +0.6),
    ],
    "old_share_high_ppol" : [
        (0.05, -0.8, -0.6),
        (0.675, -0.8, +0.6),
        (0.225, +0.8, -0.6),
        (0.05, +0.8, +0.6),
    ],
    "new_share_low_ppol" : [
        (0.05, -0.8, -0.3),
        (0.45, -0.8, +0.3),
        (0.45, +0.8, -0.3),
        (0.05, +0.8, +0.3),
    ],
    "old_share_low_ppol" : [
        (0.05, -0.8, -0.3),
        (0.675, -0.8, +0.3),
        (0.225, +0.8, -0.3),
        (0.05, +0.8, +0.3),
    ],
    "new_share_no_ppol" : [
        (0.5, -0.8, 0.0),
        (0.5, +0.8, 0.0),
    ],
    "old_share_no_ppol" : [
        (0.725, -0.8, 0.0),
        (0.275, +0.8, 0.0),
    ],
    "equal_share_high_ppol" : [
        (0.25, -0.8, -0.6),
        (0.25, -0.8, +0.6),
        (0.25, +0.8, -0.6),
        (0.25, +0.8, +0.6),
    ],
    "equal_share_low_ppol" : [
        (0.25, -0.8, -0.3),
        (0.25, -0.8, +0.3),
        (0.25, +0.8, -0.3),
        (0.25, +0.8, +0.3),
    ],
    "no_pol" : [
        (1.0, 0.0, 0.0),
    ],
}

In [5]:
workspaces = {}
fit_results = {}
for oo_name in oo_names:
    ws = {}
    fs = {}
    for name, run_conf in run_configurations.items():
        w = build_sim_ws(run_conf, input_path, oo_name, run_ab, split_helicity_reversal=split_hel)
        model = w.pdf("sim_model")
        ds = ROOT.RooStats.AsymptoticCalculator.GenerateAsimovData(model, w.set("observables"))
        fit_res = fit(w, "sim_model", ds, silent=True)
        ws[name] = w
        fs[name] = fit_res
    workspaces[oo_name] = ws
    fit_results[oo_name] = fs

[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooSimultaneous::sim_model
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooCategory::runs
[#1] INFO:Fitting -- RooAbsPdf::fitTo(sim_model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 10.9357 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_sim_model_asimovDataFullModel) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization
[#1] INFO:ObjectHandling -- RooWorkspace::import(w) importing RooSimultaneous::sim_model
[#1] INFO:ObjectHandling -- RooWorkspace::impo

In [6]:
plotter = Plotter()

In [7]:
legend_name_dict_oo = {
    "reco": "Reco, E-scheme",
    "mlvec_reco": "Reco, P-scheme",
    "clean_reco": "Reco, E-scheme, BIB-removal",
    "mlvec_clean_reco": "Reco, P-scheme, BIB-removal",
    "clean_brems_reco": "Reco, E-scheme, BIB-removal + brems",
    "mlvec_clean_brems_reco": "Reco, P-scheme, BIB-removal + brems",
    "cheat_clean_reco": "Reco, E-scheme, cheat BIB-removal",
    "mlvec_cheat_clean_reco": "Reco, P-scheme, cheat BIB-removal",
    "av_nurec_nomb_mc": "MC",
    "av_nurec_nomb_mc_rlep": "MC + Reco lep",
    "av_nurec_nomb_mc_rlep_gamma": "MC + Reco lep, iso-#gamma brems",
    "av_nurec_nomb_mc_rlep_brems": "MC + Reco lep, window brems",
    "av_nurec_nomb_mc_rlep_cheated_brems": "MC + Reco lep, cheated brems",
}

In [8]:
# for checks
plotter.draw_plots_oo_names_per_run_name("no_pol_mc", "no_pol", fit_results, oo_names=[
    "mc",
    "av_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])

In [9]:
# mlvec and BIB-removal
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_mlvec_or_not", "no_pol", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    # "mlvec_clean_brems_reco",
    # "clean_brems_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_mW_mlvec_or_not", "no_pol", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
    # "mlvec_clean_brems_reco",
    # "clean_brems_reco",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_mlvec_or_not_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_mlvec_or_not.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_mlvec_or_not_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_mlvec_or_not.pdf has been created


In [10]:
# brems recovery
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_brems_reco", "no_pol", fit_results, oo_names=[
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_mW_brems_reco", "no_pol", fit_results, oo_names=[
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_clean_brems_reco",
    "clean_brems_reco",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_brems_mc", "no_pol", fit_results, oo_names=[
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_mW_brems_mc", "no_pol", fit_results, oo_names=[
    "av_nurec_nomb_mc_rlep",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_brems_reco_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_brems_reco.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_brems_reco_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_brems_reco.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_brems_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_brems_mc.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_brems_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_brems_mc.pdf has been created


In [23]:
# reco to mc
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_reco_to_mc", "no_pol", fit_results, oo_names=[
    "reco",
    "clean_brems_reco",
    # "mlvec_clean_brems_reco",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)
plotter.draw_plots_oo_names_per_run_name("no_pol_mW_reco_to_mc", "no_pol", fit_results, oo_names=[
    "reco",
    "clean_brems_reco",
    # "mlvec_clean_brems_reco",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc",
], parameter_names=[
    "mW",
], legend_name_dict=legend_name_dict_oo, legend_pars=(0.16, 0.11, 0.68, 0.42), plot_dir=plot_dir)

Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_reco_to_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_TGC_reco_to_mc.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_reco_to_mc_no_legend.pdf has been created
Info in <TCanvas::Print>: pdf file plots/fit_results_3ab_split_hel/no_pol_mW_reco_to_mc.pdf has been created


In [12]:
plotter.draw_plots_runs_per_oo_name("mlvec_clean_reco_TGC", "mlvec_clean_reco", fit_results, parameter_names=["g1z", "ka", "la", ], legend_pars=(0.7,0.55,1.,1.))
plotter.draw_plots_runs_per_oo_name("mlvec_clean_reco_mW", "mlvec_clean_reco", fit_results, parameter_names=["mW", ], legend_pars=(0.7, 0., 1., 0.45))
plotter.draw_plots_runs_per_oo_name("mlvec_clean_reco_pols", "mlvec_clean_reco", fit_results, parameter_names=["e_pol_L", "e_pol_R", "p_pol_L", "p_pol_R"], legend_pars=(0.33,0.52,0.59,1.))

In [13]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "reco",
    "mlvec_clean_reco",
    "clean_reco",
    "mlvec_cheat_clean_reco",
    "cheat_clean_reco",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_jm", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco_jm",
    "reco_jm",
    "mlvec_clean_reco_jm",
    "clean_reco_jm",
    "mlvec_cheat_clean_reco_jm",
    "cheat_clean_reco_jm",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_jm_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_reco_jm",
    "mlvec_clean_reco",
    "mlvec_clean_reco_jm",
    "mlvec_cheat_clean_reco",
    "mlvec_cheat_clean_reco_jm",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])

In [14]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_reco_mc_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_clean_reco",
    "mlvec_clean_brems_reco",
    "mlvec_cheat_clean_reco",
    # "kinfit_clean_reco",
    "av_nurec_post94_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc",
    "av_mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_mW_reco_mc_comp", "new_share_high_ppol", fit_results, oo_names=[
    "mlvec_reco",
    "mlvec_clean_reco",
    "mlvec_clean_brems_reco",
    "mlvec_cheat_clean_reco",
    # "kinfit_clean_reco",
    "av_nurec_post94_mc",
    "av_nomb_mc_rlep",
    "av_nomb_mc",
    "av_mc",
], parameter_names=[
    "mW",
])

In [15]:
plotter.draw_plots_oo_names_per_run_name("no_pol_TGC_mc", "no_pol", fit_results, oo_names=[
    "av_nomb_mc",
    "av_mc",
    "nomb_mc",
    "mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
])

In [16]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_mW_ultracheat", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "mc",
], parameter_names=[
    "g1z",
    "ka",
    "la",
    "mW",
])
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_pols_ultracheat", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "mc",
], parameter_names=[
    "e_pol_L",
    "e_pol_R",
    "p_pol_L",
    "p_pol_R",
])

In [17]:
plotter.draw_plots_oo_names_per_run_name("new_share_high_ppol_TGC_mW_rlepcomp", "new_share_high_ppol", fit_results, oo_names=[
    "av_mc",
    "av_nurec_nomb_mc",
    "av_nurec_nomb_mc_rlep_cheated_brems",
    "av_nurec_nomb_mc_rlep_gamma",
    "av_nurec_nomb_mc_rlep_brems",
    "av_nurec_nomb_mc_rlep",
], parameter_names=[
    "g1z",
    "ka",
    "la",
    "mW",
])